# 경구약제 이미지 검출 — 최종 제출 코드

캐글 점수 **0.63420** (mAP@0.75:0.95) 을 만드는 코드다.
시행착오와 실패한 실험은 담지 않았다. 그 기록은 `알약검출_실습_v4.ipynb` 에 있다.

## 구조

```
사진  →  [1단계] YOLO 검출기 1개      "알약이 어디 있나"  1종만 구분
      →  조각 오리기 (상자 + 여백 10%)
      →  [2단계] ResNet18 분류기 9개  "무슨 약인가"      118종
      →  확률 9장을 평균  →  제출 파일
```

**검출기는 하나, 분류기는 아홉이다.** 아홉은 서로 다른 데이터·크기·훈련법으로 학습해서
같은 조각을 조금씩 다르게 판단하고, 그 평균이 각자의 실수를 지운다.
단독 최고가 0.62464인데 아홉을 합치면 0.63420이 된다.

## 실행 방법

| 섹션 | 내용 | 기본 동작 |
|---|---|---|
| 1 | 준비 | 실행 |
| 2 | 데이터 표 만들기 | **건너뜀** (코드는 읽을 수 있음) |
| 3 | 1단계 검출기 학습 | **건너뜀** |
| 4 | 2단계 분류기 9개 학습 | **건너뜀** |
| 5 | 학습된 가중치 확인 | 실행 |
| 6 | 검출 → 조각 오리기 | 실행 |
| 7 | 분류기 9개 추론 | 실행 |
| 8 | 앙상블 → 제출 파일 | 실행 |
| 9 | 제출 전 검사 | 실행 |

섹션 2~4는 `학습_실행 = False` 로 막혀 있다. 저장된 가중치가 이미 있으므로
**5번부터 실행하면 20~40분 안에 제출 파일이 나온다.**
처음부터 다시 학습하려면 `학습_실행 = True` 로 바꾸면 되지만 **하루쯤 걸린다.**

---
## 섹션 1. 준비

In [ ]:
import os, glob, time, copy, collections
import numpy as np, pandas as pd, torch, cv2
from PIL import Image
import torchvision.models as tvm
from torchvision.transforms import v2 as T
from torchvision import tv_tensors
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from ultralytics import YOLO

학습_실행 = False            # True 로 바꾸면 섹션 2~4의 학습이 실제로 돌아간다 (약 18시간)

DEV = "mps"                                          # 애플 실리콘 GPU
W0, H0 = 976, 1280                                   # 원본 사진 크기
IMGSZ, PAD = 416, 0.10                               # 검출 입력 크기 · 조각 여백 10%
MEAN = [0.485, 0.456, 0.406]                         # ImageNet 정규화 값
STD  = [0.229, 0.224, 0.225]
ROOT = "./sprint_ai_project1_data"
TE   = f"{ROOT}/test_images"                         # 캐글 test 사진
OUT  = "submissions_final"; os.makedirs(OUT, exist_ok=True)
COLS = ["annotation_id","image_id","category_id","bbox_x","bbox_y","bbox_w","bbox_h","score"]

clean = pd.read_csv("v4_clean.csv")                  # 학습 표
CLF   = sorted(set(clean["name"]))                   # 118종 — 분류기 출력 순서의 기준
C2I   = {n: i for i, n in enumerate(CLF)}            # 이름 → 분류기 번호
n2c   = dict(zip(clean["name"], clean["cat_id"]))    # 이름 → 캐글 번호
CATS  = np.array([n2c[n] for n in CLF])              # 분류기 번호 → 캐글 번호
print(f"클래스 {len(CLF)}종 · 학습 표 {len(clean):,}행 · 사진 {clean.file.nunique():,}장")

---
## 섹션 2. 데이터 표 만들기 &nbsp; `학습_실행 = True` 일 때만

AI Hub 조합 zip 8개 중 7개(`TS_2` 제외)에서 라벨을 읽어 상자 표를 만든다.
세 가지 검사를 거친다.

1. **품질 검사** — 상자 누락·중복·사진 범위 밖인 사진을 제외
2. **누수 검사** — 캐글 test와 똑같은 사진을 제거 (64×64 지문의 코사인 유사도 0.95 초과)
3. **그룹 분할** — 비슷한 장면끼리 묶어 그룹 통째로 train/val/test에 배정

전체 구현은 `scripts/rebuild_v4.py` 에 있다. 결과가 `v4_clean.csv` 다.

In [ ]:
if 학습_실행:
    os.system('"../../.venv/bin/python" scripts/rebuild_v4.py')   # → v4_clean.csv
else:
    d = pd.read_csv("v4_clean.csv")
    print("이미 만들어진 표를 사용한다")
    print(d.groupby("split").agg(사진=("file","nunique"), 상자=("file","size")).to_string())
    print(f"\n종류 {d.cat_id.nunique()}종 · 조합 {d.combo.nunique():,}개")

---
## 섹션 3. 1단계 — 검출기 학습 &nbsp; `학습_실행 = True` 일 때만

**"알약이다" 한 종류만** 구분하도록 학습한다. 무슨 약인지는 2단계가 맡는다.
찾는 일과 구분하는 일을 나누면 각자 자기 일만 잘하면 된다.

In [ ]:
if 학습_실행:
    y = YOLO("yolo11s.pt")                           # 사전학습 가중치에서 시작
    y.train(data=os.path.abspath("yolo1_data/data.yaml"),   # 클래스 1개(pill)짜리 데이터셋
            epochs=60, imgsz=IMGSZ, batch=16, device=DEV,
            project="runs/detect/runs_v3", name="yolo1_v3", exist_ok=True)
else:
    print("학습 생략 — runs/detect/runs_v3/yolo1_v3/weights/best.pt 사용")

---
## 섹션 4. 2단계 — 분류기 9개 학습 &nbsp; `학습_실행 = True` 일 때만

아홉 개는 **네 가지 레시피**에서 나온다.

| # | 이름 | 데이터 | 크롭 | 증강 | 에포크 | 전처리 |
|---|---|---|---|---|---|---|
| 1 | `v3_224` | v3 (8,898장) | 224 | 약함 | 12 | — |
| 2 | `v4_224` | v4 (11,713장) | 224 | 약함 | 12 | — |
| 3 | `bright` | v3 | 224 | 약함 | 12 | 밝기 정규화 |
| 4 | `clahe` | v3 | 224 | 약함 | 12 | CLAHE |
| 5 | `v3_384e30` | v3 | 384 | **강함** | 30 | — |
| 6~9 | `v4_384e10/20/30/40` | v4 | 384 | **강함** | 10·20·30·40 | — |

6~9번은 **한 번의 학습에서 네 지점을 저장한 것**이다.
강한 증강 때문에 에포크마다 다른 방향으로 흔들려, 같은 뿌리인데도 서로 다른 실수를 한다.

`bright` 와 `clahe` 는 단독 성적이 낮다(0.62367 · 0.61604).
그래도 넣는다 — **다르게 틀리기 때문에** 평균에서 서로를 보완한다.
빼면 점수가 떨어진다.

In [ ]:
# ---- 두 가지 전처리. 학습과 추론에 똑같이 적용해야 한다 (어긋나면 반드시 손해다) ----
_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def pp_bright(im, target=0.50):
    """사진 전체 밝기를 같은 배율로 맞춘다. 조명 차이를 지운다."""
    a = np.asarray(im, dtype=np.float32) / 255.0
    mn = a.mean()
    if mn < 1e-6: return im
    return Image.fromarray((np.clip(a * (target/mn), 0, 1) * 255).astype(np.uint8))

def pp_clahe(im):
    """LAB의 L(밝기) 채널에만 국소 대비를 키운다. 색 단서는 그대로 둔다."""
    a = cv2.cvtColor(np.asarray(im), cv2.COLOR_RGB2LAB)
    a[:, :, 0] = _clahe.apply(a[:, :, 0])
    return Image.fromarray(cv2.cvtColor(a, cv2.COLOR_LAB2RGB))

def pp_none(im): return im

### 4-1. 증강 — 같은 사진을 매번 다르게 보여주기

증강은 **한 장의 사진을 매번 조금씩 다르게 만들어 보여주는 것**이다.

```
1에포크째   005000 알약을 정면으로
2에포크째   같은 알약을 살짝 기울여서
3에포크째   같은 알약을 어둡게, 일부는 가려서
```

이렇게 해야 모델이 **"이 사진"을 외우지 않고 "이 알약"을 배운다.
특히 우리처럼 한 종류가 조합 7개(장면 7가지)뿐인 경우, 증강이 없으면
그 7장면만 완벽히 맞히고 처음 보는 장면에서 무너진다.

두 가지 세기를 썼다.

| | 224 계열 4개 | 384 계열 5개 |
|---|---|---|
| 180도 회전 | O | O |
| 색 흔들기 | ±25% | **±35%** |
| 흐리게 | O | O |
| 자르기·확대 | — | **O** |
| 기울이기·이동 | — | **O** |
| 일부 지우기 | — | **O** |

### 강한 증강의 여섯 가지

```python
T.RandomApply([T.RandomRotation((180,180))], p=0.5)
```
절반의 확률로 뒤집는다. 알약은 뒤집어 놓아도 같은 약이다.

```python
T.RandomResizedCrop(384, scale=(0.65,1.0), ratio=(0.85,1.18))
```
조각의 65~100%만 잘라 확대한다. 알약이 화면에 꽉 찰 때도, 여유 있게 놓일 때도 맞히게 한다.

```python
T.RandomAffine(degrees=10, translate=(0.05,0.05))
```
±10도 기울이고 ±5% 옮긴다. **검출기가 그린 상자가 조금 어긋나도 버티게** 하는 대비다.

```python
T.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.20, hue=0.03)
```
밝기·대비를 ±35%까지 흔든다. 다만 **색조(hue)는 ±3%로 아주 작게** 둔다 —
알약 색깔은 종류를 가르는 핵심 단서라, 많이 흔들면 흰 약과 노란 약을 헷갈리게 된다.

```python
T.RandomApply([T.GaussianBlur(3, (0.1,1.5))], p=0.3)
```
30% 확률로 흐리게. 초점이 살짝 나간 사진에 대비한다.

```python
T.RandomErasing(p=0.25, scale=(0.02,0.12))
```
**25% 확률로 조각의 2~12%를 지운다.** 여섯 중 가장 특이하다.
일부러 가려서 남은 것만으로 맞히게 훈련시키는 것이다.
가리지 않으면 "각인이 ABC네" 하나만 보고 판단하는 버릇이 들지만,
가리면 각인이 안 보일 때 모양·색·크기도 함께 보게 된다.
**한 가지 단서에 의존하지 못하게 만드는 장치다.**

### 세 가지는 세트로 묶여야 한다

강한 증강은 **크롭 384, 30에포크와 함께일 때만** 효과가 났다.

```
384 크롭만 바꿈               −0.0041
384 + 강한 증강 + 30에포크     +0.0015   ← 단독 최고 기록
```

증강이 세면 매번 다른 모습이 나오므로 **그만큼 오래 배워야** 소화한다.
그리고 잘라내고 가려도 정보가 남으려면 **조각이 커야** 한다.
하나만 떼어 쓰면 오히려 손해였다.

In [ ]:
class CropDataset(Dataset):
    """원본 사진에서 알약 하나를 오려내 학습용 조각으로 만든다."""
    def __init__(self, rows, train, crop, 강한증강, 전처리=pp_none, jitter=0.08):
        self.rows, self.crop, self.pp = rows.reset_index(drop=True), crop, 전처리
        self.jitter = jitter if train else 0
        if train and 강한증강:                        # 384 계열 — 세게 흔든다
            steps = [T.RandomApply([T.RandomRotation((180,180))], p=0.5),
                     T.RandomResizedCrop(crop, scale=(0.65,1.0), ratio=(0.85,1.18)),
                     T.RandomAffine(degrees=10, translate=(0.05,0.05)),
                     T.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.25, hue=0.03),
                     T.RandomApply([T.GaussianBlur(3,(0.1,1.5))], p=0.3),
                     T.ToDtype(torch.float32, scale=True), T.Normalize(MEAN, STD),
                     T.RandomErasing(p=0.25, scale=(0.02,0.12))]   # 일부를 가려도 맞히게
        elif train:                                   # 224 계열 — 약하게
            steps = [T.RandomApply([T.RandomRotation((180,180))], p=0.5),
                     T.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15, hue=0.02),
                     T.RandomApply([T.GaussianBlur(3,(0.1,1.0))], p=0.3),
                     T.Resize((crop,crop)),
                     T.ToDtype(torch.float32, scale=True), T.Normalize(MEAN, STD)]
        else:                                         # 검증 — 흔들지 않는다
            steps = [T.Resize((crop,crop)),
                     T.ToDtype(torch.float32, scale=True), T.Normalize(MEAN, STD)]
        self.tf = T.Compose(steps)

    def __len__(self): return len(self.rows)

    def __getitem__(self, i):
        r = self.rows.iloc[i]
        cx, cy = r.x + r.w/2, r.y + r.h/2             # 상자 중심
        w, h = r.w*(1+PAD), r.h*(1+PAD)               # 여백 10%
        if self.jitter:                               # 위치·크기를 조금씩 흔든다
            cx += np.random.uniform(-self.jitter, self.jitter) * r.w
            cy += np.random.uniform(-self.jitter, self.jitter) * r.h
            s = np.random.uniform(1-self.jitter, 1+self.jitter); w, h = w*s, h*s
        im = Image.open(IMG_PATHS[r.file]).convert("RGB").crop(
             (max(0,cx-w/2), max(0,cy-h/2), min(r.img_w,cx+w/2), min(r.img_h,cy+h/2)))
        im = self.pp(im)                              # 전처리 (모델별로 다름)
        return self.tf(tv_tensors.Image(
               torch.from_numpy(np.array(im).copy()).permute(2,0,1))), C2I[r["name"]]

In [ ]:
def 학습(표, crop, 에포크, 강한증강, 전처리, 저장경로, lr):
    """분류기 하나를 학습한다. 저장은 검증 균형 정확도가 최고일 때."""
    tr = 표[표.split == "train"]; va = 표[표.split == "val"]
    cnt = tr["name"].value_counts()
    wt  = torch.tensor([1.0/cnt[n] for n in tr["name"]], dtype=torch.double)
    dl_tr = DataLoader(CropDataset(tr, True, crop, 강한증강, 전처리), batch_size=64,
                       sampler=WeightedRandomSampler(wt, len(wt), replacement=True))
    dl_va = DataLoader(CropDataset(va, False, crop, False, 전처리), batch_size=64)

    m = tvm.resnet18(weights="DEFAULT")
    m.fc = torch.nn.Linear(m.fc.in_features, len(CLF))   # 마지막 층을 118종으로 교체
    m = m.to(DEV)
    opt = torch.optim.SGD(m.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sch = (torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100) if 강한증강
           else torch.optim.lr_scheduler.StepLR(opt, step_size=5, gamma=0.5))
    lossf = torch.nn.CrossEntropyLoss(label_smoothing=0.05)   # 정답을 100%로 가르치지 않는다

    best, state = -1, None
    for ep in range(1, 에포크+1):
        m.train()
        for x, y in dl_tr:
            x, y = x.to(DEV), y.to(DEV)
            loss = lossf(m(x), y)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 10.0); opt.step()
        sch.step()
        m.eval(); per = collections.defaultdict(lambda: [0,0])
        with torch.no_grad():
            for x, y in dl_va:
                p = m(x.to(DEV)).argmax(1).cpu()
                for t, q in zip(y.tolist(), p.tolist()):
                    per[t][1] += 1; per[t][0] += int(t == q)
        bal = float(np.mean([a/b for a, b in per.values()]))   # 종류별 정확도의 평균
        if bal > best: best, state = bal, copy.deepcopy(m.state_dict())
        print(f"    ep{ep:>2} 균형 {bal:.4f}", flush=True)
    torch.save(state, 저장경로)
    return best

In [ ]:
if 학습_실행:
    IMG_PATHS = {os.path.basename(p): os.path.abspath(p)
                 for p in glob.glob(f"{ROOT}/train_images/*.png")
                        + glob.glob("./aihub_data/**/*.png", recursive=True)}
    v3 = pd.read_csv("v3_clean.csv"); v4 = pd.read_csv("v4_clean.csv")
    학습(v3, 224, 12, False, pp_none,   "runs_v3/clf_118.pt",     0.005)   # 1
    학습(v4, 224, 12, False, pp_none,   "runs_v10/clf_v4.pt",     0.005)   # 2
    학습(v3, 224, 12, False, pp_bright, "runs_v5/clf_bright.pt",  0.005)   # 3
    학습(v3, 224, 12, False, pp_clahe,  "runs_v7/clf_clahe.pt",   0.005)   # 4
    # 5~9번은 384·강한 증강으로 길게 학습하며 10·20·30·40 에포크 지점을 저장한다
    #   scripts/train384_strong_v4.py 참조 (코사인 감쇠 기준은 100에포크)
else:
    print("학습 생략 — 저장된 가중치 9개를 사용한다")

---
## 섹션 5. 여기부터 실행 — 분류기 9개 불러오기

`.pt` 파일에는 학습으로 얻은 숫자 1,124만 개가 들어 있다. 이걸 빈 ResNet18에 부어넣는다.

In [ ]:
MODELS = [                                            # (이름, 가중치, 크롭, 전처리)
    ("v3_224",    "runs_v3/clf_118.pt",            224, pp_none),
    ("v3_384e30", "runs_v4/clf384s_ep30.pt",       384, pp_none),
    ("v4_224",    "runs_v10/clf_v4.pt",            224, pp_none),
    ("v4_384e10", "runs_v11/clf384s_v4_ep10.pt",   384, pp_none),
    ("v4_384e20", "runs_v11/clf384s_v4_ep20.pt",   384, pp_none),
    ("v4_384e30", "runs_v11/clf384s_v4_ep30.pt",   384, pp_none),
    ("v4_384e40", "runs_v11/clf384s_v4_ep40.pt",   384, pp_none),
    ("bright",    "runs_v5/clf_bright.pt",         224, pp_bright),
    ("clahe",     "runs_v7/clf_clahe.pt",          224, pp_clahe),
]
DET = "runs/detect/runs_v3/yolo1_v3/weights/best.pt"   # 1단계 검출기

print(f"{'상태':<5}{'이름':<12}{'크기':>7}  경로")
for 이름, p, crop, _ in [("검출기", DET, 0, None)] + [(n,p,c,f) for n,p,c,f in MODELS]:
    있음 = os.path.exists(p)
    mb = f"{os.path.getsize(p)//1024//1024}MB" if 있음 else "-"
    print(f"{'O' if 있음 else 'X':<5}{이름:<12}{mb:>7}  {p}")

---
## 섹션 6. 1단계 실행 — 검출하고 조각을 오린다

검출은 **한 번만** 한다. 아홉 분류기가 모두 같은 상자를 쓰기 때문에,
나중에 확률만 평균내면 된다. 상자가 서로 달랐다면 "어느 상자가 맞나"부터 정해야 한다.

In [ ]:
t0 = time.time()
y  = YOLO(DET)
te = sorted(f for f in os.listdir(TE) if f.endswith(".png"))
BOX, RAW = [], []                                     # 상자 정보 · 오려낸 조각(원본 크기)
for k in range(0, len(te), 32):
    ch = te[k:k+32]
    for f, r in zip(ch, y.predict(source=[f"{TE}/{f}" for f in ch], imgsz=IMGSZ,
                                  conf=0.01,          # 낮게 잡아 놓치는 알약을 줄인다
                                  max_det=100, device=DEV, verbose=False)):
        b, s = r.boxes.xyxy.cpu(), r.boxes.conf.cpu()
        if len(b) == 0: continue
        im = Image.open(f"{TE}/{f}").convert("RGB")
        for (x1, y1, x2, y2), sc in zip(b.tolist(), s.tolist()):
            cx, cy = (x1+x2)/2, (y1+y2)/2             # 상자 중심
            w, h = (x2-x1)*(1+PAD), (y2-y1)*(1+PAD)   # 학습 때와 같은 여백 10%
            RAW.append(im.crop((max(0,cx-w/2), max(0,cy-h/2),
                                min(W0,cx+w/2), min(H0,cy+h/2))))
            BOX.append((f, x1, y1, x2, y2, sc))
print(f"사진 {len(te)}장 → 상자 {len(BOX)}개 "
      f"(사진당 {len(BOX)/len(te):.1f}개) · {(time.time()-t0)/60:.1f}분")

---
## 섹션 7. 2단계 실행 — 분류기 9개가 각자 판단

같은 조각을 아홉 번 본다. **모델마다 자기가 학습 때 쓴 전처리와 크롭 크기를 그대로** 적용한다.
여기가 어긋나면 성능이 무너진다 — 이 프로젝트에서 가장 많이 겪은 실패 원인이다.

In [ ]:
@torch.no_grad()
def 추론(m, X, bs=64):
    return torch.cat([torch.softmax(m(X[k:k+bs].to(DEV)), 1).cpu()
                      for k in range(0, len(X), bs)]).numpy()

확률 = {}
t0 = time.time()
for 이름, ckpt, crop, pp in MODELS:
    tf = T.Compose([T.Resize((crop, crop)),
                    T.ToDtype(torch.float32, scale=True), T.Normalize(MEAN, STD)])
    X = torch.stack([tf(tv_tensors.Image(
            torch.from_numpy(np.array(pp(im)).copy()).permute(2,0,1))) for im in RAW])
    m = tvm.resnet18(); m.fc = torch.nn.Linear(m.fc.in_features, len(CLF))
    m.load_state_dict(torch.load(ckpt, map_location="cpu"))   # 학습된 숫자를 부어넣는다
    m = m.to(DEV).eval()
    확률[이름] = 추론(m, X)
    print(f"  {이름:<11} 평균 확신 {확률[이름].max(1).mean():.3f} · "
          f"예측 종류 {len(set(확률[이름].argmax(1))):>3}종 · "
          f"{(time.time()-t0)/60:.0f}분", flush=True)
    del X, m
print(f"\n총 {(time.time()-t0)/60:.1f}분")

---
## 섹션 8. 앙상블 — 확률 아홉 장을 평균내 제출 파일을 만든다

투표가 아니라 **확률의 평균**이다. 그래서 확신 있는 소수가 우물쭈물한 다수를 이길 수 있다.

제출 점수는 `검출 확신 × 분류 확률` 이다. mAP는 종류별로 점수 순서를 매겨 계산하지만,
실험해보니 이 순서를 바꿔도 결과는 거의 변하지 않았다(±0.0004). 라벨이 훨씬 중요하다.

In [ ]:
P   = np.mean(list(확률.values()), axis=0)            # 아홉 확률표의 평균
lab = P.argmax(1)                                     # 평균에서 1등인 종류

rows = []
for i, (f, x1, y1, x2, y2, dsc) in enumerate(BOX):
    bw, bh = round(x2-x1, 1), round(y2-y1, 1)
    if bw <= 0 or bh <= 0: continue                   # 반올림 뒤 넓이가 0이 된 상자는 버린다
    rows.append({"image_id": int(os.path.splitext(f)[0]),
                 "category_id": int(CATS[lab[i]]),    # 분류기 번호 → 캐글 번호
                 "bbox_x": round(x1,1), "bbox_y": round(y1,1),
                 "bbox_w": bw, "bbox_h": bh,
                 "score": round(float(dsc * P[i, lab[i]]), 5)})

sub = pd.DataFrame(rows).sort_values(["image_id","score"], ascending=[True, False])
sub.insert(0, "annotation_id", range(1, len(sub)+1))  # 1부터 매기는 고유 번호
sub[COLS].to_csv(f"{OUT}/submission.csv", index=False)
print(f"저장 {OUT}/submission.csv")
print(f"{len(sub)}행 · 사진 {sub.image_id.nunique()}장 · 종류 {sub.category_id.nunique()}종")
sub[COLS].head()

---
## 섹션 9. 제출 전 검사

캐글에 올리기 전에 형식을 확인한다. 한 번 틀리면 제출 기회를 그냥 버리게 된다.

In [ ]:
d = pd.read_csv(f"{OUT}/submission.csv")
있어야할사진 = {int(os.path.splitext(f)[0]) for f in te}
아는번호 = set(clean.cat_id)

검사 = [
    ("열 이름과 순서",      list(d.columns) == COLS),
    ("annotation_id 고유", d.annotation_id.is_unique),
    ("사진 누락 없음",      set(d.image_id) == 있어야할사진),
    ("모르는 종류 없음",    set(d.category_id) <= 아는번호),
    ("상자 폭·높이 > 0",    (d[["bbox_w","bbox_h"]] > 0).all().all()),
    ("점수 0~1",           d.score.between(0, 1).all()),
    ("좌표가 사진 안",      (d.bbox_x >= 0).all() and (d.bbox_y >= 0).all()
                            and (d.bbox_x + d.bbox_w <= W0 + 1).all()
                            and (d.bbox_y + d.bbox_h <= H0 + 1).all()),
]
for 이름, 통과 in 검사:
    print(f"  {'통과' if 통과 else '실패':<5} {이름}")
print(f"\n{sum(t for _, t in 검사)}/{len(검사)} 통과 · "
      f"{os.path.getsize(f'{OUT}/submission.csv')/1024:.0f}KB")

---
## 부록. 이 구성에 이른 근거

| 결정 | 근거 |
|---|---|
| 2단계 (검출 + 분류 분리) | 단일 단계는 두 번 시도해 두 번 다 밀렸다 (−0.0096, −0.0163) |
| 118종 전부 학습 | mAP는 정답에 있는 종류만 평균낸다. 답하지 않은 종류는 AP가 0이다 |
| 검출 해상도 416 | 640은 −0.050, 960은 −0.331. 학습 해상도와 어긋나면 무너진다 |
| 크롭 여백 10% | 학습·추론 모두 동일하게 적용해야 한다 |
| 분류기 9개 | 6개 0.62632 → 9개 0.63420. 10개 이상은 오히려 하락 |
| 단독 실패작 포함 | `clahe`(0.61604)를 빼면 점수가 떨어진다. 다르게 틀리는 게 값어치다 |
| 뼈대 전부 ResNet18 | ResNet50·EfficientNet을 넣으면 −0.0008 · −0.0042 |

전체 실험 기록과 실패 16건의 원인은 `알약검출_실습_v4.ipynb` 에 있다.